# Week 2 – LLM Client (Gemini + Ollama)
Runtime: **GPU (T4)**. Run cells top to bottom.

In [1]:
import os
os.makedirs("/content/week-2", exist_ok=True)
os.chdir("/content/week-2")   # absolute path -> safe to re-run
print(os.getcwd())

/content/week-2


In [2]:
%%bash
apt-get -qq install -y zstd > /dev/null
which ollama || curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess, time, requests

def ollama_up():
    try:
        return requests.get("http://localhost:11434", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)

print("Ollama running:", ollama_up())

Ollama running: True


In [4]:
# keep only the last lines so the progress-bar spam doesn't flood the notebook
!ollama pull llama3.2 2>&1 | tail -n 2
!ollama list

writing manifest 
success 
NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [5]:
# Warm-up: the FIRST call loads the model into memory and can take 1-2 min.
r = requests.post("http://localhost:11434/api/generate",
                  json={"model": "llama3.2", "prompt": "Say hello.",
                        "stream": False, "keep_alive": "30m"},
                  timeout=300)
print(r.json()["response"])

Hello!


In [6]:
# Works in Colab (Secrets) AND in VS Code (falls back to a hidden prompt)
import getpass
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    key = getpass.getpass("Paste GEMINI_API_KEY: ")

if not key:
    raise ValueError("No Gemini key provided")

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')
print(".env created.")

.env created.


In [7]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [8]:
!pip install -q -r requirements.txt
from google import genai
print("google-genai imported OK")

google-genai imported OK


In [ ]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests


class LLMProvider(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        ...


class GeminiProvider(LLMProvider):
    def __init__(self, api_key, model="gemini-3.5-flash-lite"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model, contents=prompt
        )
        if not response.text:
            raise RuntimeError("Gemini returned an empty response (maybe blocked).")
        return response.text


class OllamaProvider(LLMProvider):
    def __init__(self, model="llama3.2", timeout=300):
        self.model = model
        self.timeout = timeout          # was 120: too tight for cold start
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        try:
            response = requests.post(
                self.url,
                json={"model": self.model, "prompt": prompt,
                      "stream": False, "keep_alive": "30m"},
                timeout=self.timeout,
            )
            response.raise_for_status()
        except requests.exceptions.ConnectionError:
            raise RuntimeError("Ollama server not running. Re-run the server cell.")
        return response.json()["response"]


class LLMClient:
    def __init__(self, provider: LLMProvider):
        if not hasattr(provider, "generate"):
            raise TypeError("provider must implement generate()")
        self.provider = provider

    def generate(self, prompt):
        if not prompt or not prompt.strip():
            raise ValueError("Prompt cannot be empty")
        return self.provider.generate(prompt)

Writing llm_client.py


In [10]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import GeminiProvider, OllamaProvider, LLMClient

load_dotenv(override=True)

name = input("Choose provider (gemini/ollama): ").strip().lower()

if name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)
elif name == "ollama":
    provider = OllamaProvider("llama3.2")
else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)
answer = client.generate(input("Enter your question: "))
print("\nLLM Response:\n" + answer)

Writing main.py


In [11]:
%%writefile test_client.py
import pytest
from llm_client import LLMClient


class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


def test_generate():
    assert LLMClient(FakeProvider()).generate("Explain AI") == \
        "AI stands for Artificial Intelligence."


def test_empty_prompt_rejected():
    with pytest.raises(ValueError):
        LLMClient(FakeProvider()).generate("   ")


def test_bad_provider_rejected():
    with pytest.raises(TypeError):
        LLMClient(object())

Writing test_client.py


In [12]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: platformdirs-4.14.3, typeguard-4.6.0, anyio-4.16.0, langsmith-0.14.17
collected 3 items                                                              

test_client.py::test_generate PASSED                                     [ 33%]
test_client.py::test_empty_prompt_rejected PASSED                        [ 66%]
test_client.py::test_bad_provider_rejected PASSED                        [100%]

============================== 3 passed in 1.54s ===============================


In [18]:
%run main.py


LLM Response:
The "Why college?" question is bigger now than it has ever been. With rising tuition costs and alternative paths to success (like bootcamps, trade schools, and entrepreneurship), it’s smart to question the traditional route. 

To figure out if college is worth it *for you*, it helps to break it down into the core reasons people go—and the counter-arguments.

### 1. Career Preparation & Income Potential
* **The Traditional View:** A degree is often seen as a prerequisite for higher-paying jobs, corporate ladders, and specialized fields like medicine, law, engineering, or accounting.
* **The Reality:** Data consistently shows that, on average, college graduates earn significantly more over their lifetimes than those with only a high school diploma. However, this varies wildly by **major** (STEM vs. Humanities) and the **school** you attend.
* **The Counter:** For tech, trades, creative arts, and sales, portfolios, certifications, and experience often matter much more than 

In [19]:
%run main.py


LLM Response:
There are many reasons why attending college is considered a valuable investment in one's future. Here are some of the most significant benefits:

1. **Increased Earning Potential**: Studies have shown that college graduates tend to earn more than those without a degree. In fact, according to the Bureau of Labor Statistics, workers with a bachelor's degree typically earn about 50% more than those with only a high school diploma.
2. **Improved Job Prospects**: Many careers require a college degree, and having one can give you a competitive edge in the job market. College graduates are more likely to be hired, promoted, and earn higher salaries.
3. **Development of Skills and Knowledge**: College provides a structured environment where you can develop important skills such as critical thinking, problem-solving, communication, and teamwork. You'll also gain a deeper understanding of your chosen field and its applications.
4. **Networking Opportunities**: College provides a 